# ORBIT-TRUST — Space Debris Detection & Trajectory Prediction

This notebook is the ORBIT-TRUST training pipeline. It loads TLE (Two-Line Element)
data for known debris clouds and active satellites, engineers orbital features via
SGP4 propagation, visualises the feature distributions, trains a Random Forest
classifier, and saves the model artefacts used by the ORBIT-TRUST FastAPI backend.

**Sections**
1. Imports & constants
2. TLE parser
3. Sanity-check a single file
4. Build the full dataset
5. Exploratory visualisations
6. Model training & evaluation
7. Save artefacts

## 1 · Imports & constants

In [ ]:
import os
import warnings
from datetime import datetime, timezone

import joblib
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe in notebooks too
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 — registers the 3-D projection
from scipy.stats import gaussian_kde
from sgp4.api import Satrec, jday
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, train_test_split

warnings.filterwarnings('ignore')   # suppress minor deprecation noise

# Physical constants
EARTH_RADIUS_KM = 6371.0          # mean Earth radius in km
MU_KM3_S2       = 398600.4418     # Earth's gravitational parameter  km³/s²

# Paths — notebook lives in Spacemain/
NOTEBOOK_DIR = os.path.abspath('')
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'data')
MODELS_DIR   = os.path.join(NOTEBOOK_DIR, 'backend', 'models')

print('Data dir  :', DATA_DIR)
print('Models dir:', MODELS_DIR)

## 2 · TLE parser

Each TLE file contains triplets: `name / line-1 / line-2`.  
We propagate each satellite to the current UTC epoch with SGP4 and derive
20 orbital features plus a binary label (1 = debris, 0 = active satellite).

In [ ]:
def parse_tle_file(filename: str, label: int) -> list[dict]:
    """
    Read a TLE file from DATA_DIR, propagate every object to the current UTC
    time, and return a list of feature dictionaries.

    Parameters
    ----------
    filename : str   — file name inside DATA_DIR (e.g. 'active-sate.txt')
    label    : int   — 1 for debris, 0 for active satellite

    Returns
    -------
    list[dict]  — one dict per successfully propagated object
    """
    filepath = os.path.join(DATA_DIR, filename)
    records  = []

    with open(filepath, 'r') as fh:
        lines = fh.readlines()

    # Current UTC epoch — timezone-aware to avoid DeprecationWarning
    now = datetime.now(timezone.utc)
    jd, fr = jday(now.year, now.month, now.day,
                  now.hour, now.minute, now.second)

    for i in range(0, len(lines), 3):
        if i + 2 >= len(lines):   # incomplete triplet at end of file
            break

        line1 = lines[i + 1].strip()
        line2 = lines[i + 2].strip()

        try:
            sat = Satrec.twoline2rv(line1, line2)
            err, r, v = sat.sgp4(jd, fr)

            # Skip objects with propagation errors or NaN position/velocity
            if err != 0 or any(x != x for x in list(r) + list(v)):
                continue

            # --- Derived orbital quantities ---
            # Semi-major axis from mean motion (Kepler's third law)
            # no_kozai is in rad/min; convert to rad/s for the formula
            n_rad_s  = sat.no_kozai / 60.0          # rad/s
            sma      = (MU_KM3_S2 / n_rad_s**2) ** (1.0 / 3.0)  # km

            ecc      = sat.ecco
            dist     = np.sqrt(r[0]**2 + r[1]**2 + r[2]**2)   # km from Earth centre
            vel      = np.sqrt(v[0]**2 + v[1]**2 + v[2]**2)   # km/s
            soe      = (vel**2) / 2.0 - (MU_KM3_S2 / dist)    # specific orbital energy

            records.append({
                'eccentricity':            ecc,
                'inclination':             sat.inclo,
                'raan':                    sat.nodeo,
                'arg_perigee':             sat.argpo,
                'mean_anomaly':            sat.mo,
                'mean_motion':             sat.no_kozai,
                'semi_major_axis':         sma,
                'x':  r[0], 'y':  r[1], 'z':  r[2],
                'vx': v[0], 'vy': v[1], 'vz': v[2],
                'altitude':                sma - EARTH_RADIUS_KM,
                'perigee':                 sma * (1.0 - ecc),
                'apogee':                  sma * (1.0 + ecc),
                'orbital_period':          86400.0 / sat.no_kozai,
                'distance_from_center':    dist,
                'specific_orbital_energy': soe,
                'velocity_magnitude':      vel,
                'label':                   label,
            })
        except Exception:
            continue   # silently skip malformed TLE lines

    return records

print('Parser defined.')

## 3 · Sanity-check a single file

Verify that the SGP4 library can parse every TLE triplet in one of the debris
files before we process the full dataset.

In [ ]:
from sgp4.earth_gravity import wgs72
from sgp4.io import twoline2rv

check_file = os.path.join(DATA_DIR, 'cosmos-1408-debris.txt')
ok_count   = 0

with open(check_file, 'r') as fh:
    lines = fh.readlines()

for i in range(0, len(lines), 3):
    if i + 2 >= len(lines):
        break
    s = lines[i + 1].strip()
    t = lines[i + 2].strip()
    assert twoline2rv(s, t, wgs72), f'Parse failed at line {i}'
    ok_count += 1

print(f'Sanity check passed — {ok_count} TLE objects verified in cosmos-1408-debris.txt')

## 4 · Build the full dataset

In [ ]:
DEBRIS_FILES = [
    'cosmos-1408-debris.txt',
    'cosmos-2251-debris.txt',
    'fengyun-1c-debris.txt',
    'iridium-33-debris.txt',
]
ACTIVE_FILES = ['active-sate.txt']

all_rows = []

for fname in DEBRIS_FILES:
    rows = parse_tle_file(fname, label=1)
    all_rows.extend(rows)
    print(f'  {fname:35s}  →  {len(rows):5d} debris objects')

for fname in ACTIVE_FILES:
    rows = parse_tle_file(fname, label=0)
    all_rows.extend(rows)
    print(f'  {fname:35s}  →  {len(rows):5d} active satellites')

df = pd.DataFrame(all_rows).dropna()

n_debris = (df['label'] == 1).sum()
n_active = (df['label'] == 0).sum()
print(f'\nTotal rows : {len(df)}')
print(f'  Debris   : {n_debris}')
print(f'  Active   : {n_active}')
df.describe()

## 5 · Exploratory visualisations

In [ ]:
# Inclination vs Eccentricity scatter — debris tends to have higher eccentricity
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='inclination', y='eccentricity', hue='label', alpha=0.5)
plt.title('Inclination vs Eccentricity')
plt.xlabel('Inclination (rad)')
plt.ylabel('Eccentricity')
plt.legend(title='Label', labels=['Active Satellite', 'Debris'])
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, 'images', 'incli_vs_ecc.png'), dpi=100)
plt.show()

In [ ]:
# Semi-major axis distribution — debris clouds cluster at specific altitudes
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='semi_major_axis', hue='label', kde=True, bins=30, alpha=0.6)
plt.title('Distribution of Semi-Major Axis')
plt.xlabel('Semi-Major Axis (km)')
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, 'images', 'sma_dist.png'), dpi=100)
plt.show()

In [ ]:
# Specific orbital energy — a useful discriminator between debris and active sats
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='specific_orbital_energy', hue='label', kde=True, bins=50, alpha=0.6)
plt.title('Distribution of Specific Orbital Energy')
plt.xlabel('Specific Orbital Energy (km²/s²)')
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, 'images', 'soe.png'), dpi=100)
plt.show()

In [ ]:
# Mean motion distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='mean_motion', hue='label', kde=True, bins=100, alpha=0.6)
plt.title('Distribution of Mean Motion')
plt.xlabel('Mean Motion (rad/min)')
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, 'images', 'mm.png'), dpi=100)
plt.show()

In [ ]:
# 3-D scatter of the three most informative orbital parameters
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')
sc  = ax.scatter(
    df['inclination'], df['eccentricity'], df['semi_major_axis'],
    c=df['label'], cmap='viridis', alpha=0.4, s=2
)
ax.set_xlabel('Inclination (rad)')
ax.set_ylabel('Eccentricity')
ax.set_zlabel('Semi-Major Axis (km)')
plt.title('3-D Scatter of Orbital Parameters')
plt.legend(*sc.legend_elements(), title='Class')
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, 'images', 'scatter.png'), dpi=100)
plt.show()

## 6 · Model training & evaluation

The dataset is class-imbalanced (more active satellites than debris).  
We apply **SMOTE** on the training split to balance the classes before fitting
a Random Forest classifier.

In [ ]:
FEATURE_COLS = [c for c in df.columns if c != 'label']

X = df[FEATURE_COLS]
y = df['label']

# 80 / 20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Oversample the minority class (debris) in the training set only
X_bal, y_bal = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f'After SMOTE — class counts: {dict(zip(*np.unique(y_bal, return_counts=True)))}')

# Train Random Forest with OOB scoring enabled
clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    n_jobs=-1,
)
clf.fit(X_bal, y_bal)

# Evaluate on the held-out test set
y_pred = clf.predict(X_test)
print(f'\nTest Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'OOB Score     : {clf.oob_score_:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Active Satellite', 'Debris']))

In [ ]:
# 5-fold cross-validation on the balanced training data for a more robust estimate
cv_scores = cross_val_score(clf, X_bal, y_bal, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-fold CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Feature importance bar chart
importances = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(10).plot(kind='bar')
plt.title('Top-10 Feature Importances')
plt.ylabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, 'images', 'feature_importance.png'), dpi=100)
plt.show()

print('\nAll feature importances:')
print(importances.to_string())

## 7 · Save artefacts

The trained model and the ordered feature list are saved to
`backend/models/` so the FastAPI server can load them at startup.

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(clf,          os.path.join(MODELS_DIR, 'rf_model.joblib'))
joblib.dump(FEATURE_COLS, os.path.join(MODELS_DIR, 'features.joblib'))

print(f'Model saved  → {os.path.join(MODELS_DIR, "rf_model.joblib")}')
print(f'Features saved → {os.path.join(MODELS_DIR, "features.joblib")}')